In [37]:
import pandas as pd
import numpy as np
import default_risk.config as cfg
import dtale
import gc

bureau_df = pd.read_parquet(cfg.CLEANS_DIR / "bureau_train-cleaned.parquet")

In [38]:
bureau_balance_agg= pd.read_parquet(cfg.PROCESSED_DIR / "bureau_balance_train-processed.parquet")
bureau_df=bureau_df.merge(bureau_balance_agg,how="left",on="id_bureau")
bureau_df['has_bureau_balance_data'] = bureau_df['balance_months_balance_min'].notna().astype(int)

In [39]:
bureau_df.head()

,id_curr,id_bureau,credit_active,credit_currency,days_credit,flag_have_credit_day_overdue,days_credit_enddate,days_credit_enddate_first_cluster_values,days_credit_enddate_is_missing,days_credit_enddate_closed,...,balance_months_balance_min,balance_months_balance_max,balance_status_score_max,balance_status_score_mean,balance_status_score_std,balance_status_0_mean,balance_status_0_sum,balance_is_delincuency_mean,balance_is_delincuency_sum,has_bureau_balance_data
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,215354,5714463,Active,currency 1,-208,0,1075.0,1075.0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,215354,5714464,Active,currency 1,-203,0,528.0,528.0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,215354,5714466,Active,currency 1,-629,0,1197.0,1197.0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [40]:
bureau_df["ratio_credit_annuity"]= np.where(bureau_df["amt_annuity"] !=0, bureau_df["amt_credit_sum"] / bureau_df["amt_annuity"] , np.nan )  
bureau_df["completetitud_ratio"] = np.where(bureau_df["amt_credit_sum_debt"]!=0,bureau_df["amt_credit_sum"] / bureau_df["amt_credit_sum_debt"],np.nan)
bureau_df["ratio_debt_limit"] =  np.where(bureau_df["amt_credit_sum"] !=0, bureau_df["amt_credit_sum_limit"] / bureau_df["amt_credit_sum"], np.nan )  
bureau_df["log_amt_credit_sum"] = np.log1p(bureau_df["amt_credit_sum"])

bureau_df["credit_active"]=bureau_df["credit_active"].str.lower()
bureau_df= pd.get_dummies(bureau_df, columns=["credit_active"], dtype=int)

bureau_df.sort_values(["id_curr", "days_credit"],inplace=True,ascending=False)
last_two = bureau_df.groupby("id_curr").head(1)
last_two = last_two.copy()
last_two["loan_order"] = last_two.groupby("id_curr").cumcount() + 1



last_two_columns = last_two.drop(columns=["id_bureau"]).pivot(index="id_curr", columns="loan_order")
last_two_columns.columns = [f"bureau_{col[0]}_loan_{col[1]}" for col in last_two_columns.columns]
last_two_columns= last_two_columns.reset_index()


bureau_df["credit_type"]=bureau_df["credit_type"].str.lower()
bureau_df= pd.get_dummies(bureau_df,columns= ["credit_type"])


In [41]:
bureau_agg_dic_active= {
    "id_curr": ["count"],

    #monetary
    "amt_credit_sum": ["max", "mean","sum","std"],
    "amt_credit_sum_limit": ["max","mean","min","std"],
    "amt_annuity" : ["max","mean","min","std"], #
    "amt_credit_sum_debt" : ["max","mean","sum","std"],

    #log_transformed
    "log_amt_credit_sum": ["mean","std"],

    #counters
    "cnt_credit_prolong": ["max","mean"], #,"sum"
    "days_credit_update": ["min","max","mean"], 
    "days_credit": ["min","max","mean"], 
    "days_credit_enddate": ["max","mean"], 

    "ratio_credit_annuity" : ["max","mean","min"],
    "completetitud_ratio" : ["mean","min"],
    "amt_annuity_is_missing" : ["mean","sum"],
    "have_amt_credit_sum_overdue" : ["mean","sum"],
    "amt_credit_max_overdue" :["max","mean","sum"],

    #categorical
    "credit_type_credit card" : ["sum"], # ,"sum"
    "credit_type_mortgage" : ["mean","sum"], # ,"sum"
    "credit_type_microloan" : ["mean","sum"], #  ,"sum"
    "credit_type_consumer credit": ["sum"], #,"sum"
    
}

In [42]:

bureau_agg_dic_closed = {
    "id_curr": ["count"],

    #monetary
    "amt_credit_sum": ["max", "mean","sum","std"],
    "amt_credit_sum_limit": ["max","mean","min"], #,"std"
    "amt_annuity" : ["max","mean","min","std"],
    "amt_credit_sum_debt" : ["max","mean","sum","std"],

    #log_transformed
    "log_amt_credit_sum": ["mean","std"],

    #counters
    "days_credit_update": ["min","max","mean"], 
    "days_credit": ["min","max","mean"], 
    "days_enddate_fact": ["max"], 

    "ratio_credit_annuity" : ["max","mean","min"],
    "completetitud_ratio" : ["mean","min"],
    "credit_active_sold" : ["mean","sum"],
    "amt_annuity_is_missing" : ["mean","sum"],
    "amt_credit_max_overdue" :["max","mean","sum"],
    "amt_credit_max_overdue_is_missing" :["mean","sum"],

    #categorical
    "credit_type_credit card" : ["sum"], #,"sum"
    "credit_type_mortgage" : ["mean","sum"], #,"sum"
    "credit_type_microloan" : ["mean","sum"], #,"sum"
    "credit_type_consumer credit": ["sum"], #,"sum"
}


In [43]:

bureau_agg_dic= {
    "id_curr": ["count"],

    #monetary
    "amt_credit_sum": ["max", "mean","sum","std"],
    "amt_credit_sum_limit": ["max","mean","min","std"],
    "amt_annuity" : ["max","mean","min","std"],
    "amt_credit_sum_debt" : ["max","mean","sum","std"],

    #log_transformed
    "log_amt_credit_sum": ["mean","std"],

    #counters
    "cnt_credit_prolong": ["max","mean","sum"],
    "days_credit_update": ["min","max","mean"], 
    "days_credit": ["min","max","mean"], 
    "days_enddate_fact": ["max"], 

    #other
    "ratio_credit_annuity" : ["max","mean","min"],
    "completetitud_ratio" : ["mean","min"],
    
    #categoricals
    "has_bureau_balance_data": ["sum", "mean"],
    "credit_active_sold" : ["mean","sum"],
    "amt_credit_sum_debt_is_negative": ["mean","sum"],
    "days_enddate_fact_is_missing" : ["mean","sum"],
    "have_amt_credit_sum_overdue" : ["mean","sum"],
    "amt_credit_sum_limit_is_missing" : ["mean","sum"],
    "amt_credit_sum_limit_is_zero" : ["mean","sum"],
    "amt_annuity_is_missing" : ["mean","sum"],
}


In [44]:
agg_from_bureau_balance_dict = {
    "balance_status_score_max": ["max"], #0.76 
    "balance_months_balance_max": ["max"], #0.76 
    "balance_months_balance_min": ["min"], #0.76 
    "balance_months_since_delincuency" : ["max"],#0.76 
    "balance_is_delincuency_sum" : ["max"],#0.76 
    "balance_is_delincuency_mean" : ["mean"],#0.76 
}

In [45]:
active_loans= bureau_df [bureau_df["credit_active_active"] == 1]
closed_but_recent= (bureau_df["credit_active_active"] != 1) & (bureau_df["days_enddate_fact"] > -5080)
non_active_loans=  bureau_df[closed_but_recent]
bureau_active_aggregated = active_loans.groupby("id_curr").agg(bureau_agg_dic_active|agg_from_bureau_balance_dict ).add_prefix("active_")
bureau_non_active_aggregated = non_active_loans.groupby("id_curr").agg(bureau_agg_dic_closed|agg_from_bureau_balance_dict ).add_prefix("closed_")
bureau_aggregated= bureau_active_aggregated.merge(bureau_non_active_aggregated,how="outer",on="id_curr")

bureau_aggregated.columns= [f"{col[0]}_{col[1]}" for col in bureau_aggregated.columns]
bureau_aggregated= bureau_aggregated.reset_index()


time_window_df= pd.read_parquet(cfg.PROCESSED_DIR / "bureau_balance_time_window.parquet")
bureau_final_df= last_two_columns.merge(bureau_aggregated,how="left",on="id_curr")

bureau_final_df.to_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

In [ ]:
dtale.show(bureau_final_df)

: 

In [ ]:
bureau_balance_agg= pd.read_parquet(cfg.PROCESSED_DIR / "bureu_balance_agg.parquet")